# Классификаторы кухни 1

**ML4 · Classification** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

В этом уроке вы будете использовать набор данных, который сохранили в прошлом уроке, полный сбалансированных, чистых данных о кухнях.

Вы будете использовать этот набор данных с различными классификаторами, чтобы _предсказать национальную кухню по набору ингредиентов_. При этом вы узнаете больше о некоторых способах использования алгоритмов для задач классификации.


# Подготовка

Если вы завершили [Урок 1](https://github.com/microsoft/ML-For-Beginners/blob/main/4-Classification/1-Introduction/README.md), убедитесь, что файл _cleaned_cuisines.csv_ существует в корневой папке `/data` для этих четырёх уроков.



## Упражнение – предсказать национальную кухню

1. Работая в папке _notebook.ipynb_ этого урока, импортируйте этот файл вместе с библиотекой Pandas:

    ```python
    import pandas as pd
    cuisines_df = pd.read_csv("../data/cleaned_cuisines.csv")
    cuisines_df.head()
    ```

    Данные выглядят так:

|     | Unnamed: 0 | cuisine | almond | angelica | anise | anise_seed | apple | apple_brandy | apricot | armagnac | ... | whiskey | white_bread | white_wine | whole_grain_wheat_flour | wine | wood | yam | yeast | yogurt | zucchini |
| --- | ---------- | ------- | ------ | -------- | ----- | ---------- | ----- | ------------ | ------- | -------- | --- | ------- | ----------- | ---------- | ----------------------- | ---- | ---- | --- | ----- | ------ | -------- |
| 0   | 0          | indian  | 0      | 0        | 0     | 0          | 0     | 0            | 0       | 0        | ... | 0       | 0           | 0          | 0                       | 0    | 0    | 0   | 0     | 0      | 0        |
| 1   | 1          | indian  | 1      | 0        | 0     | 0          | 0     | 0            | 0       | 0        | ... | 0       | 0           | 0          | 0                       | 0    | 0    | 0   | 0     | 0      | 0        |
| 2   | 2          | indian  | 0      | 0        | 0     | 0          | 0     | 0            | 0       | 0        | ... | 0       | 0           | 0          | 0                       | 0    | 0    | 0   | 0     | 0      | 0        |
| 3   | 3          | indian  | 0      | 0        | 0     | 0          | 0     | 0            | 0       | 0        | ... | 0       | 0           | 0          | 0                       | 0    | 0    | 0   | 0     | 0      | 0        |
| 4   | 4          | indian  | 0      | 0        | 0     | 0          | 0     | 0            | 0       | 0        | ... | 0       | 0           | 0          | 0                       | 0    | 0    | 0   | 0     | 1      | 0        |
  

1. Теперь импортируйте ещё несколько библиотек:

    ```python
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,classification_report, precision_recall_curve
    from sklearn.svm import SVC
    import numpy as np
    ```

1. Разделите координаты X и y на два датафрейма для обучения. `cuisine` может быть датафреймом с метками:

    ```python
    cuisines_label_df = cuisines_df['cuisine']
    cuisines_label_df.head()
    ```

    Это будет выглядеть так:

    ```output
    0    indian
    1    indian
    2    indian
    3    indian
    4    indian
    Name: cuisine, dtype: object
    ```

1. Удалите столбцы `Unnamed: 0` и `cuisine`, вызвав метод `drop()`. Сохраните оставшиеся данные как признаки для обучения:

    ```python
    cuisines_feature_df = cuisines_df.drop(['Unnamed: 0', 'cuisine'], axis=1)
    cuisines_feature_df.head()
    ```

    Ваши признаки выглядят так:

|      | almond | angelica | anise | anise_seed | apple | apple_brandy | apricot | armagnac | artemisia | artichoke |  ... | whiskey | white_bread | white_wine | whole_grain_wheat_flour | wine | wood |  yam | yeast | yogurt | zucchini |
| ---: | -----: | -------: | ----: | ---------: | ----: | -----------: | ------: | -------: | --------: | --------: | ---: | ------: | ----------: | ---------: | ----------------------: | ---: | ---: | ---: | ----: | -----: | -------: |
|    0 |      0 |        0 |     0 |          0 |     0 |            0 |       0 |        0 |         0 |         0 |  ... |       0 |           0 |          0 |                       0 |    0 |    0 |    0 |     0 |      0 |        0 | 0 |
|    1 |      1 |        0 |     0 |          0 |     0 |            0 |       0 |        0 |         0 |         0 |  ... |       0 |           0 |          0 |                       0 |    0 |    0 |    0 |     0 |      0 |        0 | 0 |
|    2 |      0 |        0 |     0 |          0 |     0 |            0 |       0 |        0 |         0 |         0 |  ... |       0 |           0 |          0 |                       0 |    0 |    0 |    0 |     0 |      0 |        0 | 0 |
|    3 |      0 |        0 |     0 |          0 |     0 |            0 |       0 |        0 |         0 |         0 |  ... |       0 |           0 |          0 |                       0 |    0 |    0 |    0 |     0 |      0 |        0 | 0 |
|    4 |      0 |        0 |     0 |          0 |     0 |            0 |       0 |        0 |         0 |         0 |  ... |       0 |           0 |          0 |                       0 |    0 |    0 |    0 |     0 |      1 |        0 | 0 |

Теперь вы готовы обучить вашу модель!



## Выбор классификатора

Теперь, когда ваши данные чистые и готовы к обучению, вам нужно решить, какой алгоритм использовать для задачи.

Scikit-learn объединяет классификацию под категорией Обучение с учителем, и в этой категории вы найдете много способов классифицировать. [Разнообразие](https://scikit-learn.org/stable/supervised_learning.html) сначала может ошеломить. Следующие методы включают техники классификации:

- Линейные модели
- Машины опорных векторов
- Стохастический градиентный спуск
- Ближайшие соседи
- Гауссовские процессы
- Деревья решений
- Ансамблевые методы (голосование классификаторов)
- Многоклассовые и многоцелевые алгоритмы (многоклассовая и многометочная классификация, многоклассово-многоцелевой классификатор)

> Вы также можете использовать [нейронные сети для классификации данных](https://scikit-learn.org/stable/modules/neural_networks_supervised.html#classification), но это вне рамок данного урока.

### Какой классификатор выбрать?

Итак, какой классификатор выбрать? Часто эффективным способом является запуск нескольких и поиск хорошего результата. Scikit-learn предлагает [побочное сравнение](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) на созданном наборе данных, сравнивая KNeighbors, SVC двумя способами, GaussianProcessClassifier, DecisionTreeClassifier, RandomForestClassifier, MLPClassifier, AdaBoostClassifier, GaussianNB и QuadraticDiscriminantAnalysis, показывая результаты визуализированными:

![сравнение классификаторов](ms-assets/ml/comparison.edfab56193a85e7f.webp)
> Графики из документации Scikit-learn

> AutoML решает эту задачу аккуратно, запуская эти сравнения в облаке, позволяя вам выбрать лучший алгоритм для ваших данных. Попробуйте [здесь](https://learn.microsoft.com/learn/modules/automate-model-selection-with-azure-automl/?WT.mc_id=academic-77952-leestott)

### Лучший подход

Лучшим методом, чем слепое угадывание, является следование идеям из этого загружаемого [справочника по ML](https://learn.microsoft.com/azure/machine-learning/algorithm-cheat-sheet?WT.mc_id=academic-77952-leestott). Здесь мы узнаём, что для нашей задачи с многоклассовой классификацией у нас есть несколько вариантов:

![шпаргалка по многоклассовым задачам](ms-assets/ml/cheatsheet.07a475ea444d2223.webp)
> Раздел справочника Microsoft по алгоритмам, описывающий варианты многоклассовой классификации

✅ Скачайте этот справочник, распечатайте и повесьте на стену!

### Рассуждения

Давайте попробуем рассуждать по поводу разных подходов с учётом имеющихся ограничений:

- **Нейронные сети слишком тяжёлые**. Учитывая наш чистый, но небольшой набор данных и тот факт, что обучение проводится локально в ноутбуках, нейронные сети слишком ресурсоёмки для этой задачи.
- **Нет двухклассового классификатора**. Логистическая регрессия изначально работает с двумя классами; для нескольких классов применяется схема one-vs-rest (OvR) или multinomial — ниже мы используем `ovr`.
- **Дерево решений или логистическая регрессия могут подойти**. Дерево решений может подойти, или логистическая регрессия для многоклассовых данных.
- **Многоклассовые бустинговые деревья решают другую задачу**. Они более подходящи для непараметрических задач, например, для построения ранжирований, что нам не нужно.

### Использование Scikit-learn

Мы будем использовать Scikit-learn для анализа наших данных. Однако существует много способов использовать логистическую регрессию в Scikit-learn. Посмотрите параметры, которые можно передать [здесь](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html?highlight=logistic%20regressio#sklearn.linear_model.LogisticRegression).

В основном, есть два важных параметра — `multi_class` и `solver` — которые нужно указать, когда вы просите Scikit-learn выполнить логистическую регрессию. Значение `multi_class` задаёт определённое поведение. Значение `solver` указывает, какой алгоритм использовать. Не все решения (solvers) совместимы со всеми значениями `multi_class`.

Согласно документации, в случае многоклассовой задачи алгоритм обучения:

- **Использует схему one-vs-rest (OvR)**, если опция `multi_class` установлена в `ovr`
- **Использует функцию потерь кросс-энтропии**, если `multi_class` установлена в `multinomial`. (В настоящее время опция `multinomial` поддерживается только решателями ‘lbfgs’, ‘sag’, ‘saga’ и ‘newton-cg’)."

> 🎓 Здесь 'scheme' может быть 'ovr' (один-против-остальных) или 'multinomial'. Поскольку логистическая регрессия изначально предназначена для бинарной классификации, эти схемы позволяют лучше обрабатывать многоклассовые задачи. [источник](https://machinelearningmastery.com/one-vs-rest-and-one-vs-one-for-multi-class-classification/)

> 🎓 'Solver' определяется как "алгоритм, используемый для решения задачи оптимизации". [источник](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html?highlight=logistic%20regressio#sklearn.linear_model.LogisticRegression).

Scikit-learn предлагает эту таблицу, чтобы объяснить, как решатели справляются с различными типами данных:

![решатели](ms-assets/ml/solvers.5fc648618529e627.webp)



## Упражнение – разделение данных

Мы можем сосредоточиться на логистической регрессии для первой попытки обучения, так как вы недавно изучали её в предыдущем уроке.
Разделите данные на обучающую и тестовую выборки, вызвав `train_test_split()`:



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

## Упражнение – применить логистическую регрессию

Так как вы используете многоклассовый случай, нужно выбрать, какую _схему_ использовать и какой _решатель_ задать. Используйте LogisticRegression с многоклассовой настройкой и решателем **liblinear** для обучения.

1. Создайте логистическую регрессию с параметром multi_class, установленным в `ovr`, и решателем, установленным в `liblinear`:

    ```python
    lr = LogisticRegression(multi_class='ovr',solver='liblinear')
    model = lr.fit(X_train, np.ravel(y_train))
    
    accuracy = model.score(X_test, y_test)
    print ("Accuracy is {}".format(accuracy))
    ```

    ✅ Попробуйте другой решатель, например `lbfgs`, который часто используется по умолчанию

    > Обратите внимание, используйте функцию Pandas [`ravel`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.ravel.html) для уплощения данных, когда это необходимо.

    Точность хорошая — более **80%**!

1. Вы можете увидеть работу модели, проверив одну строку данных (#50):

    ```python
    print(f'ingredients: {X_test.iloc[50][X_test.iloc[50]!=0].keys()}')
    print(f'cuisine: {y_test.iloc[50]}')
    ```

    Результат выводится:

   ```output
   ingredients: Index(['cilantro', 'onion', 'pea', 'potato', 'tomato', 'vegetable_oil'], dtype='object')
   cuisine: indian
   ```

   ✅ Попробуйте другой номер строки и проверьте результаты
1. Вникая глубже, вы можете проверить точность этого предсказания:

    ```python
    test= X_test.iloc[50].values.reshape(-1, 1).T
    proba = model.predict_proba(test)
    classes = model.classes_
    resultdf = pd.DataFrame(data=proba, columns=classes)
    
    topPrediction = resultdf.T.sort_values(by=[0], ascending = [False])
    topPrediction.head()
    ```

    Результат напечатан – индийская кухня является лучшим предположением с высокой вероятностью:

    |          |        0 |
    | -------: | -------: |
    |   indian | 0.715851 |
    |  chinese | 0.229475 |
    | japanese | 0.029763 |
    |   korean | 0.017277 |
    |     thai | 0.007634 |

    ✅ Можете ли вы объяснить, почему модель достаточно уверена, что это индийская кухня?

1. Получите больше информации, распечатав отчет о классификации, как вы делали в уроках по регрессии:

    ```python
    y_pred = model.predict(X_test)
    print(classification_report(y_test,y_pred))
    ```

    |              | precision | recall | f1-score | support |
    | ------------ | --------- | ------ | -------- | ------- |
    | chinese      | 0.73      | 0.71   | 0.72     | 229     |
    | indian       | 0.91      | 0.93   | 0.92     | 254     |
    | japanese     | 0.70      | 0.75   | 0.72     | 220     |
    | korean       | 0.86      | 0.76   | 0.81     | 242     |
    | thai         | 0.79      | 0.85   | 0.82     | 254     |
    | accuracy     |           |        | 0.80     | 1199    |
    | macro avg    | 0.80      | 0.80   | 0.80     | 1199    |
    | weighted avg | 0.80      | 0.80   | 0.80     | 1199    |



## Задача

В этом уроке вы использовали очищенные данные для построения модели машинного обучения, которая может предсказывать национальную кухню на основе набора ингредиентов. Найдите время, чтобы изучить множество вариантов, которые предоставляет Scikit-learn для классификации данных. Глубже разберитесь в концепции «решателя» (solver), чтобы понять, что происходит за кулисами.



## Обзор и самостоятельное изучение

Углубитесь в математику, лежащую в основе логистической регрессии, в [этом уроке](https://people.eecs.berkeley.edu/~russell/classes/cs194/f11/lectures/CS194%20Fall%202011%20Lecture%2006.pdf)



## Задание

[Изучите решатели](https://github.com/microsoft/ML-For-Beginners/blob/main/4-Classification/2-Classifiers-1/assignment.md)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — k-nearest-neighbors classifier

●●○ · ~15 min

> kNN is the simplest non-parametric classifier: find the k training points closest to the query (Euclidean distance) and take a majority vote of their labels. Two tie rules make it deterministic: equal distances are resolved by smaller training index (use a stable sort), and equal vote counts are resolved by the smallest class label (`np.bincount(...).argmax()` does exactly that).

Implement `knn_predict(X_train, y_train, X_test, k)` → array of predicted labels for each row of `X_test`.

- distance = Euclidean, `((X_train - x) ** 2).sum(1)` is enough (no need to sqrt);
- pick the `k` smallest distances — on ties prefer the smaller training index (`np.argsort(d, kind='stable')`);
- majority vote — on ties prefer the smallest label (`np.bincount(votes).argmax()`).

<details><summary>Hint</summary>

Per test row: idx = np.argsort(d, kind='stable')[:k]; pred = np.bincount(y_train[idx], minlength=n_classes).argmax().

</details>


In [ ]:
import numpy as np

def knn_predict(X_train, y_train, X_test, k):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=int)
    X_test = np.asarray(X_test, dtype=float)
    preds = []
    # your code — loop over test rows
    return np.array(preds)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, m, d = int(rng.integers(8, 25)), int(rng.integers(2, 8)), int(rng.integers(2, 5))\n    c = int(rng.integers(2, 5))\n    X_train = rng.normal(0, 2, (n, d))\n    y_train = rng.integers(0, c, n)\n    X_test = rng.normal(0, 2, (m, d))\n    k = int(rng.integers(1, 8))\n    if rng.random() < 0.35 and m >= 1:\n        # guaranteed distance tie: duplicate a row under a different label,\n        # drop a test point exactly on it — the smaller train index must win\n        i, j = sorted(int(x) for x in rng.choice(n, 2, replace=False))\n        X_train[j] = X_train[i]\n        y_train[j] = (int(y_train[i]) + 1) % c\n        X_test[0] = X_train[i]\n        k = 1\n    return {\"X_train\": X_train, \"y_train\": y_train, \"X_test\": X_test, \"k\": k}\n\ndef knn_predict(X_train, y_train, X_test, k):\n    X_train = np.asarray(X_train, dtype=float)\n    y_train = np.asarray(y_train, dtype=int)\n    X_test = np.asarray(X_test, dtype=float)\n    n_classes = int(y_train.max()) + 1\n    preds = []\n    for x in X_test:\n        dist = ((X_train - x) ** 2).sum(axis=1)\n        idx = np.argsort(dist, kind=\"stable\")[:k]\n        preds.append(int(np.bincount(y_train[idx], minlength=n_classes).argmax()))\n    return np.array(preds)", "knn_predict", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/4-Classification/2-Classifiers-1)
- [Учебник ML (ШАД): обучение с учителем](https://contest.yandex.ru/tracks/ml/supervised-learning/introduction)
- [Учебник ML (ШАД): метрики и кросс-валидация](https://contest.yandex.ru/tracks/ml/model-evaluation/introduction)
- [Математика для анализа данных: теория вероятностей](https://contest.yandex.ru/tracks/math/probability-theory/chapter-overview)